# QRBSA First-Test Scalar-Order Debug (Trained)

This notebook reproduces the QRBSA forward path on the first test sample:

1. Load the first test LR/HR quaternion pair.
2. Convert LR from scalar-first `(w, x, y, z)` to scalar-last `(x, y, z, w)`.
3. Load the trained QRBSA checkpoint and run the model.
4. Convert SR back from scalar-last to scalar-first.
5. Save the debug arrays to an `.npz` in `debugging/`.
6. Reload the `.npz` and render LR / SR / HR with the visualization helper.

Note: this notebook uses `experiments/IN718/qrbsa_01/logs/run_config.json` for dataset/model hyperparameters and loads `experiments/IN718/qrbsa_01/checkpoints/best_model.pt`.

In [ ]:
import json
import os
import sys
from pathlib import Path
from types import SimpleNamespace

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

repo_root = Path.cwd()
if not (repo_root / "training").exists():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "training").exists():
            repo_root = p
            break

for p in [repo_root, repo_root / "Q-RBSA", repo_root / "Q-RBSA" / "model"]:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import Image, display

from training.data_loading import build_dataloader
from qrbsa_1d import QRBSA_1D
from utils.symmetry_utils import resolve_symmetry
from visualization.visualize_sr_results import render_sr_hr_lr_side_by_side


In [ ]:
exp_dir = repo_root / "experiments" / "IN718" / "qrbsa_01"
debug_dir = repo_root / "debugging"
debug_dir.mkdir(parents=True, exist_ok=True)

config_path = exp_dir / "logs" / "run_config.json"
if not config_path.exists():
    config_path = exp_dir / "config.json"

model_mode = "trained_checkpoint"
checkpoint_path = exp_dir / "checkpoints" / "best_model.pt"
npz_path = debug_dir / "qrbsa_first_test_scalar_order_trained_debug.npz"
png_path = debug_dir / "qrbsa_first_test_scalar_order_trained_debug_lr_sr_hr_ipf.png"

with open(config_path, "r") as f:
    run_cfg = json.load(f)

dataset_root = run_cfg["dataset_root"]
dataset_info_path = Path(dataset_root) / "dataset_info.json"
with open(dataset_info_path, "r") as f:
    dataset_info = json.load(f)

symmetry_str = run_cfg.get("symmetry", dataset_info.get("symmetry", "Oh"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("repo_root:", repo_root)
print("config_path:", config_path)
print("model_mode:", model_mode)
print("checkpoint_path:", checkpoint_path)
print("dataset_root:", dataset_root)
print("symmetry:", symmetry_str)
print("device:", device)

def to_chw(q: torch.Tensor) -> torch.Tensor:
    if q.dim() == 4 and q.shape[-1] == 4:
        return q.permute(0, 3, 1, 2).contiguous()
    return q.contiguous()

def to_hwc_numpy(q: torch.Tensor) -> np.ndarray:
    q = q.detach().cpu().float()
    if q.dim() == 3 and q.shape[0] == 4:
        return q.permute(1, 2, 0).numpy()
    return q.numpy()

def scalar_first2last(q: torch.Tensor) -> torch.Tensor:
    return torch.cat([q[:, 1:], q[:, :1]], dim=1)

def scalar_last2first(q: torch.Tensor) -> torch.Tensor:
    return torch.cat([q[:, -1:], q[:, :-1]], dim=1)

loader = build_dataloader(
    dataset_root,
    split="Test",
    batch_size=1,
    shuffle=False,
    num_workers=0,
    seed=run_cfg.get("seed", 42),
)
first_lr_path, first_hr_path = loader.dataset.pairs[0]
print("first_lr_path:", first_lr_path)
print("first_hr_path:", first_hr_path)


In [ ]:
model = QRBSA_1D(
    SimpleNamespace(
        n_colors=4,
        n_resblocks=run_cfg.get("n_resblocks", 16),
        n_feats=run_cfg.get("n_feats", 64),
        scale=run_cfg.get("scale", 4),
    )
).to(device)
ckpt = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

batch = next(iter(loader))
lr_chw_wxyz = to_chw(batch[0]).to(device=device, dtype=torch.float32)
hr_chw_wxyz = to_chw(batch[1]).to(device=device, dtype=torch.float32)

# Keep the network path CHW throughout.
# We do exactly one HWC -> CHW conversion on load, and exactly one CHW -> HWC
# conversion later when saving / plotting arrays.
assert lr_chw_wxyz.ndim == 4 and lr_chw_wxyz.shape[1] == 4
assert hr_chw_wxyz.ndim == 4 and hr_chw_wxyz.shape[1] == 4

with torch.no_grad():
    lr_chw_xyzw = scalar_first2last(lr_chw_wxyz)
    assert lr_chw_xyzw.shape == lr_chw_wxyz.shape

    sr_chw_xyzw = model(lr_chw_xyzw)
    sr_chw_wxyz = F.normalize(scalar_last2first(sr_chw_xyzw), p=2, dim=1)

    sr_chw_xyzw_conj = model(lr_chw_xyzw.conj())
    sr_chw_wxyz_conj = F.normalize(scalar_last2first(sr_chw_xyzw_conj), p=2, dim=1)

    assert sr_chw_xyzw.ndim == 4 and sr_chw_xyzw.shape[1] == 4
    assert sr_chw_wxyz.ndim == 4 and sr_chw_wxyz.shape[1] == 4
    assert sr_chw_wxyz.shape[2:] == hr_chw_wxyz.shape[2:]

# Convert to HWC only here, after the network path is complete.
lr_wxyz = to_hwc_numpy(lr_chw_wxyz[0]).astype(np.float32, copy=False)
lr_xyzw = to_hwc_numpy(lr_chw_xyzw[0]).astype(np.float32, copy=False)
sr_xyzw = to_hwc_numpy(sr_chw_xyzw[0]).astype(np.float32, copy=False)
sr_wxyz = to_hwc_numpy(sr_chw_wxyz[0]).astype(np.float32, copy=False)
sr_xyzw_conj = to_hwc_numpy(sr_chw_xyzw_conj[0]).astype(np.float32, copy=False)
sr_wxyz_conj = to_hwc_numpy(sr_chw_wxyz_conj[0]).astype(np.float32, copy=False)
hr_wxyz = to_hwc_numpy(hr_chw_wxyz[0]).astype(np.float32, copy=False)

np.savez_compressed(
    npz_path,
    lr_wxyz=lr_wxyz,
    lr_xyzw=lr_xyzw,
    sr_xyzw=sr_xyzw,
    sr_wxyz=sr_wxyz,
    sr_xyzw_conj=sr_xyzw_conj,
    sr_wxyz_conj=sr_wxyz_conj,
    hr_wxyz=hr_wxyz,
    sample_index=np.int64(0),
    model_mode=np.array(model_mode),
    checkpoint_path=np.array(str(checkpoint_path)),
    symmetry=np.array(symmetry_str),
    first_lr_path=np.array(str(first_lr_path)),
    first_hr_path=np.array(str(first_hr_path)),
    config_path=np.array(str(config_path)),
)

print("Saved debug NPZ:", npz_path)
print("lr_wxyz shape:", lr_wxyz.shape)
print("lr_xyzw shape:", lr_xyzw.shape)
print("sr_xyzw shape:", sr_xyzw.shape)
print("sr_wxyz shape:", sr_wxyz.shape)
print("hr_wxyz shape:", hr_wxyz.shape)


In [ ]:
lr_chw_wxyz.shape, hr_chw_wxyz.shape, sr_chw_wxyz.shape

In [ ]:
debug_npz = np.load(npz_path)
loaded_model_mode = str(debug_npz["model_mode"])
loaded_checkpoint_path = str(debug_npz["checkpoint_path"])
loaded_symmetry = str(debug_npz["symmetry"])
loaded_sym_class = resolve_symmetry(loaded_symmetry)

lr_wxyz = debug_npz["lr_wxyz"]
sr_wxyz = debug_npz["sr_wxyz"]
sr_xyzw = debug_npz["sr_xyzw"]
sr_wxyz_conj = debug_npz["sr_wxyz_conj"]
sr_xyzw_conj = debug_npz["sr_xyzw_conj"]
hr_wxyz = debug_npz["hr_wxyz"]

print("Loaded from NPZ:")
print("  model_mode:", loaded_model_mode)
print("  checkpoint_path:", loaded_checkpoint_path)
print("  symmetry:", loaded_symmetry)
print("  first_lr_path:", str(debug_npz["first_lr_path"]))
print("  first_hr_path:", str(debug_npz["first_hr_path"]))
print("  lr_wxyz:", lr_wxyz.shape)
print("  sr_wxyz:", sr_wxyz.shape)
print("  sr_xyzw:", sr_xyzw.shape)
print("  sr_wxyz_conj:", sr_wxyz_conj.shape)
print("  sr_xyzw_conj:", sr_xyzw_conj.shape)
print("  hr_wxyz:", hr_wxyz.shape)


In [ ]:
# SR_WXYZ

render_sr_hr_lr_side_by_side(
    sr_q_arr=sr_wxyz,
    hr_q_arr=hr_wxyz,
    lr_q_arr=lr_wxyz,
    sym_class=loaded_sym_class,
    out_png=str(png_path),
    ref_dir="ALL",
    include_key=True,
    overwrite=True,
    format_input=True,
    dpi=300,
)

print("Saved debug PNG:", png_path)
display(Image(filename=str(png_path)))


In [ ]:
# SR_XYZW

render_sr_hr_lr_side_by_side(
    sr_q_arr=sr_xyzw,
    hr_q_arr=hr_wxyz,
    lr_q_arr=lr_wxyz,  
    sym_class=loaded_sym_class,
    out_png=str(png_path.with_name(png_path.stem + "_xyzw.png")),
    ref_dir="ALL",
    include_key=True,
    overwrite=True,
    format_input=True,
    dpi=300,
)   

print("Saved debug PNG:", png_path.with_name(png_path.stem + "_xyzw.png"))
display(Image(filename=str(png_path.with_name(png_path.stem + "_xyzw.png"))))


In [ ]:
# SR_WXYZ_conj

render_sr_hr_lr_side_by_side(
    sr_q_arr=sr_wxyz_conj,
    hr_q_arr=hr_wxyz,
    lr_q_arr=lr_wxyz,  
    sym_class=loaded_sym_class,
    out_png=str(png_path.with_name(png_path.stem + "_wxyz_conj.png")),
    ref_dir="ALL",
    include_key=True,
    overwrite=True,
    format_input=True,
    dpi=300,
)   

print("Saved debug PNG:", png_path.with_name(png_path.stem + "_wxyz_conj.png"))
display(Image(filename=str(png_path.with_name(png_path.stem + "_wxyz_conj.png"))))

In [ ]:
# SR_XYZW_conj

render_sr_hr_lr_side_by_side(
    sr_q_arr=sr_xyzw_conj,
    hr_q_arr=hr_wxyz,
    lr_q_arr=lr_wxyz,  
    sym_class=loaded_sym_class,
    out_png=str(png_path.with_name(png_path.stem + "_xyzw_conj.png")),
    ref_dir="ALL",
    include_key=True,
    overwrite=True,
    format_input=True,
    dpi=300,
)   

print("Saved debug PNG:", png_path.with_name(png_path.stem + "_xyzw_conj.png"))
display(Image(filename=str(png_path.with_name(png_path.stem + "_xyzw_conj.png"))))

## Convention Note

The cells above are for scalar-order and layout debugging. The production QRBSA training path is now interpreted as:

- Dataset LR / HR quaternions are passive, scalar-first `(w, x, y, z)`.
- Before QRBSA, LR is conjugated to active convention and then reordered to scalar-last `(x, y, z, w)`.
- After QRBSA, SR is reordered back to active scalar-first `(w, x, y, z)`.
- HR is also conjugated to active convention for the training loss, so SR and HR are compared in the same convention.
- SR is conjugated back to passive only for visualization / export, so plots remain in the repo's usual EBSD convention.

Important distinction:

- QRBSA kernels are quaternion-valued weights, but the standard QRBSA convolution path applies plain left Hamilton multiplication `W ⊗ x`.
- That is not the same as a physical rotation action `W ⊗ x ⊗ W^{-1}`.
- So there are two separate issues: scalar order / active-passive convention for orientation data, and left-multiply quaternion feature mixing inside the network.

In other words, the model can use quaternion-valued filters without those filters themselves being interpreted as physical crystal-rotation operators.